In [50]:
import torch
from torch.utils.data import DataLoader, random_split, TensorDataset
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [51]:
import urllib.request

# Download Tiny Shakespeare directly into your workspace
# url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
file_path = "tinyshakespeare.txt"

# urllib.request.urlretrieve(url, file_path)

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

print(f"Dataset length: {len(text)} characters")
print("Sample text:\n", text[:150])

Dataset length: 1115394 characters
Sample text:
 First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

A


In [ ]:
itos, stoi = dict(), dict() 

words = text.split(' ')
unique_words = sorted(list(set(text.split(' '))))
vocab_size = len(unique_words)

# stoi and itos mappings 
stoi = {w: i for (i, w) in enumerate(unique_words)}
itos = {i: w for (i, w) in enumerate(unique_words)}

data = torch.tensor([stoi[w] for w in words], dtype=torch.long)

# We need to create "chunks of text" 
seq_len = 64
n = data.shape[0]
X_list, Y_list = list(), list()
for i in range(0, len(data) - seq_len, seq_len):
    x = data[i: i + seq_len]
    y = data[i + 1: i + seq_len + 1]

    X_list.append(x)
    Y_list.append(y)


X = torch.stack(X_list, dim=0)
Y = torch.stack(Y_list, dim=0)


batch_size = 32
dataset = TensorDataset(X, Y)
train_size = int(.80*len(dataset))
test_size = int(.10*len(dataset))
val_size = len(dataset) - train_size - test_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, test_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  drop_last=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,  drop_last=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,  drop_last=True, pin_memory=True)


def tensor_to_sentence(token_tensor):
    if not isinstance(token_tensor, torch.Tensor):
        token_tensor = torch.tensor(token_tensor, dtype=torch.long)

    token_tensor = token_tensor.detach().cpu()

    if token_tensor.dim() == 0:
        return itos.get(int(token_tensor.item()), "<UNK>")

    if token_tensor.dim() == 1:
        return " ".join(itos.get(int(idx), "<UNK>") for idx in token_tensor.tolist())

    if token_tensor.dim() == 2:
        return [
            " ".join(itos.get(int(idx), "<UNK>") for idx in row.tolist())
            for row in token_tensor
        ]

    raise ValueError("Expected a 0D, 1D, or 2D tensor of token ids.")

print(f"X tensor shape {X.shape}")
print(f"Y tensor shape {Y.shape}")
print(f"vocab size:  {vocab_size}")
assert torch.equal(X[:, 1:seq_len], Y[:, :seq_len - 1]), "Label != Next Token"

print(f"len(train_loader): {len(train_loader)}")
print(f"len(val_loader): {len(val_loader)}")
print(f"len(test_loader): {len(test_loader)}")



print(train_size + test_size + val_size, len(dataloader))


X tensor shape torch.Size([2654, 64])
Y tensor shape torch.Size([2654, 64])
vocab size:  42197
2654 82


In [ ]:



train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

    logging.info(f"Training set size: {len(train_dataset)}")
    logging.info(f"Validation set size: {len(val_dataset)}")
    logging.info(f"Test set size: {len(test_dataset)}")

    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)
    